# Euler-Lagrange Equations

This notebook contains the programmatic verification for the **Euler-Lagrange Equations** entry from the THEORIA dataset.

**Entry ID:** euler_lagrange_equations  
**Required Library:** sympy 1.13.1

## Description
The Euler-Lagrange equations are second-order differential equations that determine the equations of motion for a mechanical system from its Lagrangian. They arise from requiring the action functional to be stationary under arbitrary variations of the path. Any trajectory satisfying these equations represents a physically realizable motion of the system. This formulation is equivalent to Newton's laws but offers advantages for constrained systems and generalizes naturally to field theories.

## Installation
First, let's install the required library:

In [ ]:
# Install required library with exact version
!pip install sympy==1.13.1

## Programmatic Verification

The following code verifies the derivation mathematically:

In [ ]:
import sympy as sp

# =====================================================
# Programmatic verification: Euler-Lagrange Equations
#
# This mirrors the derivation steps:
#  - Step 1: Define the action functional S as integral of L
#  - Steps 2-3: Consider path variations with fixed endpoints
#  - Steps 4-5: Expand Lagrangian and compute first variation
#  - Steps 6-7: Integration by parts, boundary terms vanish
#  - Steps 8-9: Apply fundamental lemma of calculus of variations
#  - Step 10: Obtain Euler-Lagrange equations
# =====================================================

# Define time variable and symbolic functions
t = sp.Symbol('t', real=True)
t1, t2 = sp.symbols('t_1 t_2', real=True)
epsilon = sp.Symbol('epsilon', real=True)

# For a single degree of freedom demonstration
# q(t) is the generalized coordinate
q = sp.Function('q')
eta = sp.Function('eta')  # variation function

# L is a general Lagrangian function L(q, dq/dt, t)
L = sp.Function('L')

# ---------------------------
# Step 1: Action functional
# S[q] = integral from t1 to t2 of L(q, dq/dt, t) dt
# ---------------------------
q_t = q(t)
dq_dt = sp.diff(q(t), t)

# The action is S = integral of L(q(t), q'(t), t) dt
# We represent this symbolically
L_original = L(q_t, dq_dt, t)

# ---------------------------
# Step 2: Path variation
# q(t) -> q(t) + epsilon*eta(t) with eta(t1) = eta(t2) = 0
# ---------------------------
eta_t = eta(t)
deta_dt = sp.diff(eta(t), t)

# Varied path
q_varied = q_t + epsilon * eta_t
dq_varied = dq_dt + epsilon * deta_dt

# ---------------------------
# Step 3-4: Expand Lagrangian to first order in epsilon
# L(q + eps*eta, dq + eps*deta, t) = L(q, dq, t) + eps*(dL/dq * eta + dL/d(dq) * deta) + O(eps^2)
# ---------------------------

# Define partial derivatives of L symbolically
# We use auxiliary symbols for the partial derivatives
dL_dq = sp.Function('dL_dq')  # partial L / partial q
dL_ddq = sp.Function('dL_ddq')  # partial L / partial (dq/dt)

# First variation of Lagrangian (first order in epsilon)
delta_L = dL_dq(q_t, dq_dt, t) * eta_t + dL_ddq(q_t, dq_dt, t) * deta_dt

# ---------------------------
# Step 5: First variation of action
# delta S = integral from t1 to t2 of (dL/dq * eta + dL/d(dq) * deta) dt
# ---------------------------

# The variation of the action is:
# delta_S = Integral(delta_L, (t, t1, t2))

# ---------------------------
# Step 6-7: Integration by parts on the second term
# integral(dL/d(dq) * deta/dt) dt = [dL/d(dq) * eta]_t1^t2 - integral(d/dt(dL/d(dq)) * eta) dt
# Boundary terms vanish since eta(t1) = eta(t2) = 0
# ---------------------------

# After integration by parts, the variation becomes:
# delta_S = integral from t1 to t2 of (dL/dq - d/dt(dL/d(dq))) * eta dt

# Define the time derivative of dL/d(dq)
d_dt_dL_ddq = sp.Function('d_dt_dL_ddq')  # d/dt(partial L / partial (dq/dt))

# The integrand after integration by parts
integrand_after_ibp = (dL_dq(q_t, dq_dt, t) - d_dt_dL_ddq(q_t, dq_dt, t)) * eta_t

# ---------------------------
# Step 8-9: Fundamental lemma of calculus of variations
# If integral(f(t) * eta(t)) dt = 0 for all eta(t) vanishing at endpoints,
# then f(t) = 0 throughout the interval
# ---------------------------

# Therefore: dL/dq - d/dt(dL/d(dq)) = 0
# Or equivalently: d/dt(dL/d(dq)) - dL/dq = 0

# ---------------------------
# Step 10: Verify Euler-Lagrange equation with a concrete example
# Use L = T - V for a simple harmonic oscillator: L = (1/2)*m*dq^2 - (1/2)*k*q^2
# ---------------------------

m, k = sp.symbols('m k', positive=True, real=True)
q_sym = sp.Function('q')(t)
dq_sym = sp.diff(q_sym, t)

# Lagrangian for simple harmonic oscillator
L_sho = sp.Rational(1, 2) * m * dq_sym**2 - sp.Rational(1, 2) * k * q_sym**2

# Compute partial derivatives
partial_L_partial_q = sp.diff(L_sho, q_sym)
partial_L_partial_dq = sp.diff(L_sho, dq_sym)

# Verify partial derivatives
assert sp.simplify(partial_L_partial_q + k * q_sym) == 0, "partial L/partial q should be -k*q"
assert sp.simplify(partial_L_partial_dq - m * dq_sym) == 0, "partial L/partial dq should be m*dq"

# Compute d/dt(partial L / partial dq)
d_dt_partial_L_partial_dq = sp.diff(partial_L_partial_dq, t)

# This equals m * d^2q/dt^2
d2q_dt2 = sp.diff(q_sym, t, 2)
assert sp.simplify(d_dt_partial_L_partial_dq - m * d2q_dt2) == 0, "d/dt(partial L/partial dq) should be m*d2q/dt2"

# ---------------------------
# Verify Euler-Lagrange equation gives correct equation of motion
# d/dt(partial L/partial dq) - partial L/partial q = 0
# => m*d2q/dt2 - (-k*q) = 0
# => m*d2q/dt2 + k*q = 0 (simple harmonic oscillator equation)
# ---------------------------

euler_lagrange_lhs = d_dt_partial_L_partial_dq - partial_L_partial_q
expected_eom = m * d2q_dt2 + k * q_sym

assert sp.simplify(euler_lagrange_lhs - expected_eom) == 0, "Euler-Lagrange should give m*d2q/dt2 + k*q = 0"

# ---------------------------
# Additional verification: Free particle
# L = (1/2)*m*dq^2, should give m*d2q/dt2 = 0
# ---------------------------

L_free = sp.Rational(1, 2) * m * dq_sym**2

partial_L_free_q = sp.diff(L_free, q_sym)
partial_L_free_dq = sp.diff(L_free, dq_sym)
d_dt_partial_L_free_dq = sp.diff(partial_L_free_dq, t)

euler_lagrange_free = d_dt_partial_L_free_dq - partial_L_free_q

# Should equal m*d2q/dt2 = 0 for free particle
assert sp.simplify(euler_lagrange_free - m * d2q_dt2) == 0, "Free particle EL should give m*d2q/dt2"

# ---------------------------
# Verification: Particle in gravitational field
# L = (1/2)*m*dq^2 - m*g*q, should give m*d2q/dt2 = -m*g
# ---------------------------

g = sp.Symbol('g', positive=True, real=True)
L_grav = sp.Rational(1, 2) * m * dq_sym**2 - m * g * q_sym

partial_L_grav_q = sp.diff(L_grav, q_sym)
partial_L_grav_dq = sp.diff(L_grav, dq_sym)
d_dt_partial_L_grav_dq = sp.diff(partial_L_grav_dq, t)

euler_lagrange_grav = d_dt_partial_L_grav_dq - partial_L_grav_q

# Should equal m*d2q/dt2 + m*g = 0, i.e., d2q/dt2 = -g
expected_grav_eom = m * d2q_dt2 + m * g
assert sp.simplify(euler_lagrange_grav - expected_grav_eom) == 0, "Gravitational EL should give m*d2q/dt2 + m*g"

# ---------------------------
# Verify integration by parts identity symbolically
# integral(f * dg/dt) dt = [f*g] - integral(df/dt * g) dt
# ---------------------------

f = sp.Function('f')(t)
g_func = sp.Function('g')(t)

# d/dt(f*g) = df/dt * g + f * dg/dt
product_rule = sp.diff(f * g_func, t)
expected_product_rule = sp.diff(f, t) * g_func + f * sp.diff(g_func, t)

assert sp.simplify(product_rule - expected_product_rule) == 0, "Product rule verification"

# This confirms: f * dg/dt = d/dt(f*g) - df/dt * g
# Integrating: integral(f * dg/dt) = [f*g] - integral(df/dt * g)
# Which is the integration by parts formula used in Step 6

print("All Euler-Lagrange equation derivation steps verified successfully!")


## Source

📖 **View this entry:** [theoria-dataset.org/entries.html?entry=euler_lagrange_equations.json](https://theoria-dataset.org/entries.html?entry=euler_lagrange_equations.json)

This verification code is part of the [THEORIA dataset](https://github.com/theoria-dataset/theoria-dataset), a curated collection of theoretical physics derivations with programmatic verification.

**License:** CC-BY 4.0